### CCD Model Prediction based on the input formulated

In [1]:
# Python v3.6.8
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

In [2]:
# Input dataset (From Formulation)
data = {
    'A': [400,500,358.579,500,600,641.421,500,500,600,500,500,500,400],
    'B': [250,200,200,200,150,200,270.711,200,250,129.289,200,200,150],
    'Disintegration Time': [38,35,40,37,39,42,36,34,38,40,36,35,39],
    'Drug Content': [97.5,98.2,96.8,97.7,96.5,97.0,98.5,96.9,99.0,97.2,98.0,97.8,96.7],
    'Folding Endurance': [235,245,225,240,230,220,250,248,240,225,245,238,230],
    'Thickness': [0.9,1.0,0.8,1.0,0.9,1.1,1.0,1.0,1.1,0.7,1.0,1.0,0.8],
    'Surface pH': [6.7,6.8,6.6,6.8,6.7,6.9,6.8,6.7,6.9,6.5,6.8,6.7,6.6],
}
df = pd.DataFrame(data)

In [3]:
# Features and targets from formulated dataset
inputVariables = df[['A', 'B']]
outputVariables = df.drop(columns=['A', 'B'])

### Train Data:
MultiOutputRegressor Model Allows predicting multiple targets at once.
### Ridge:
A regularized linear regression model that prevents overfitting, suitable for small datasets.

In [4]:
model = MultiOutputRegressor(Ridge(alpha=1.0))
model.fit(inputVariables, outputVariables)

MultiOutputRegressor(estimator=Ridge())

##### HPMC (A): 400 to 600 → a 200 range
##### PEG (B): 150 to 250 → a 100 range

##### [Grid Size: 25 × 25 | ~8 mg, ~4 mg | 625 Combos]
##### [Grid Size: 50 × 50	| ~4 mg, ~2 mg | 2500 Combos]
##### [Grid Size: 100 × 100 | ~2 mg, ~1 mg | 10,000 Combos]

In [5]:
# Generate Grid
A_range = np.linspace(400, 600,1000)
B_range = np.linspace(150, 250, 1000)
grid = np.array([[a, b] for a in A_range for b in B_range])

In [6]:
# Predict
predictions = model.predict(grid)

In [7]:
# Combine results
results = pd.DataFrame(grid, columns=['A', 'B'])
outputs = pd.DataFrame(predictions, columns = outputVariables.columns)
full = pd.concat([results, outputs], axis=1)

In [8]:
# Filter results
filtered = full[
    (full['Disintegration Time'] >= 30) & (full['Disintegration Time'] <= 37) &
    (full['Drug Content'] >= 97.8) & (full['Drug Content'] <= 99.0) &
    (full['Folding Endurance'] >= 240) & (full['Folding Endurance'] <= 255) &
    (full['Thickness'] >= 0.8) & (full['Thickness'] <= 1.1) &
    (full['Surface pH'] >= 6.6) & (full['Surface pH'] <= 6.8)
].copy()

In [9]:
print(full)

            A         B  Disintegration Time  Drug Content  Folding Endurance  \
0       400.0  150.0000            38.218893     96.682948         230.195554   
1       400.0  150.1001            38.216977     96.684234         230.208155   
2       400.0  150.2002            38.215061     96.685520         230.220755   
3       400.0  150.3003            38.213145     96.686805         230.233356   
4       400.0  150.4004            38.211229     96.688091         230.245957   
...       ...       ...                  ...           ...                ...   
999995  600.0  249.5996            37.019540     98.358063         242.215582   
999996  600.0  249.6997            37.017624     98.359348         242.228182   
999997  600.0  249.7998            37.015708     98.360634         242.240783   
999998  600.0  249.8999            37.013792     98.361920         242.253384   
999999  600.0  250.0000            37.011876     98.363206         242.265985   

        Thickness  Surface 

In [10]:
print(filtered)

                 A           B  Disintegration Time  Drug Content  \
869     400.000000  236.986987            36.553862     97.800341   
870     400.000000  237.087087            36.551946     97.801626   
871     400.000000  237.187187            36.550030     97.802912   
872     400.000000  237.287287            36.548114     97.804198   
873     400.000000  237.387387            36.546198     97.805484   
...            ...         ...                  ...           ...   
541838  508.308308  233.883884            36.996182     97.974771   
541839  508.308308  233.983984            36.994266     97.976057   
542837  508.508509  233.783784            36.998806     97.973881   
542838  508.508509  233.883884            36.996890     97.975167   
543837  508.708709  233.783784            36.999513     97.974277   

        Folding Endurance  Thickness  Surface pH  
869            241.145639   0.922589    6.707205  
870            241.158240   0.922771    6.707386  
871            241

In [11]:
# Scoring
filtered['FE_Diff'] = abs(filtered['Folding Endurance'] - 250)
filtered['pH_Diff'] = abs(filtered['Surface pH'] - 6.7)
filtered['Score'] = filtered['FE_Diff'] + filtered['pH_Diff']

In [12]:
# Select top results
top5 = filtered.sort_values(by='Score').head(5)
top5 = top5[['A', 'B', 'Disintegration Time', 'Drug Content', 'Folding Endurance', 'Thickness', 'Surface pH']]

In [13]:
print(top5)

               A      B  Disintegration Time  Drug Content  Folding Endurance  \
999   400.000000  250.0            36.304778     97.967499         242.783742   
1999  400.200200  250.0            36.305486     97.967895         242.783224   
2999  400.400400  250.0            36.306193     97.968292         242.782705   
3999  400.600601  250.0            36.306901     97.968688         242.782187   
4999  400.800801  250.0            36.307609     97.969084         242.781669   

      Thickness  Surface pH  
999    0.946150    6.730766  
1999   0.946331    6.730947  
2999   0.946513    6.731128  
3999   0.946694    6.731309  
4999   0.946875    6.731491  


In [14]:
# Save to file
output_path = "top5_optimal_formulations.csv"
top5.to_csv(output_path, index=False)